# Sourced from Calplus (https://github.com/Calplus)
<!-- Sourced from Calplus (https://github.com/Calplus) -->

# The Great Sorter

### Setting Things Up

Before we can proceed, we have to import modules and clean up the data set given to us, to make the data more manageable. 

Here are the modules we imported and why: 
+ The regular expression (re) module provides support for working with regular expressions - patterns used to match strings or parts of strings
+ The statistics modules is used to compute the standard deviation for each category (i.e. School, CGPA, Gender).
+ The matplotlib.pyplot module allows us to visualise our data using graphs
+ The IPython.core.display module allows 

In [ ]:
# Sourced from Calplus (https://github.com/Calplus)
import re                        # to parse initial .csv file.
import statistics                # to calculate median and pstdev.
import matplotlib.pyplot as plt  # to visualise data using graphs.
import os                        # to access file path
from IPython.core.display import SVG
import time 

We also added colours and boldness to our texts and code! We did this so that the code will be more readable throughout the whole notebook.

In [ ]:
# Sourced from Calplus (https://github.com/Calplus)
# formatting text
BLUE = "\033[1;94m"
RESET = "\033[0m" 
RED = '\033[1;91m'
GREEN = '\033[1;96m'
BOLD = "\033[1m"

For the verbose boolean variable – this can be set to True for us to see the intermediate steps that our code takes to reach the final result. Moreover, this is very useful to see which part of our code went wrong, and we can debug our code easily.

In [ ]:
# Sourced from Calplus (https://github.com/Calplus)
# enable this to see verbose output (for debugging)
verbose = False
recordRuntime = time.time() # start recording runtime

### Application of Computational Thinking

#### Decomposition
How did we decompose this complex problem?
1) Firstly, to sort the students out, we need to make our data more managable. We do this by encoding the data since working with numbers  and comparing them are easier than strings.
2) Now, we need to decide how to sort the students based on the different categories. To do this, we must decide which category to prioritise in each TG based on the frequencies and variations of values in each category.
3) After calculating the frequency -> We can calculate the PSTDEV -> Can create a priority list of the 3 categories (high to low PSTDEV, because the higher the STDEV, the more imbalanced the category).
4) Now that we know which category to prioritize, we will see the probability of each unique value in the categories so that we can see which group is more common (since we want to spread them out first). This generates a list of requirements from the "most perfect" to the "least perfect" matches.
5) Lastly, we can start sorting the people into groups using the requirement list.

In [ ]:
# Sourced from Calplus (https://github.com/Calplus)
display(SVG(filename='assets/tree.svg'))

#### Pattern Recognition
+ Some functions can be reused in other larger processes due to their nature. For instance, we use encodeDataset() in createFullDataset(), prioritiser(),  and fetchTGroupInfo() because all of them need datasets to be set up before they can be processed. computeFrequenciesOfDictionary() is also used in two different functions, countCategoricalFrequencies() and extractDistributions() since both processes, although used in different parts of the algorithm, require frequency datasets.

+ We can compare the spread of data in each category using STDEV. One possible issue we might face using this method could be the fact that, while Gender and CGPA can take two possible values each, School can take up 18 different values.

To explain this point, let's take the example of schools and gender. A high stddev in schools may reflect a few overrepresented schools, but the same standard deviation in gender could suggest extreme gender imbalance!

However, while inspecting the current dataset of 6000 students we were given, the max possible standard deviations lay in the normalised range (out of 1) of 0.2 - 0.303. For schools, the greatest standard deviation achievable was 11.45, while for cgpa and gender, it was 25. Neither of these values were reached, nor did they come close to the maximum standard deviations.

This proves that PSDTEV is a great and simple contender for cross-categorical comparison. This will

+ Probabilities of unique categorical values can be used for intra-categorical (within the category) comparison. This can tell us which kind of student to allocate first, and which one to allocate next, and so on.

#### Abstraction
+ All large processes in the algorithm have been modularised into smaller components as shown in each overarching flowchart.
+ Every lower level component is independent and can be reused if another higher-level function ever requires it. However, in some cases, they work as helper functions as well. Some of the more independent functions carry out data retrieval and low-level computations.
+ For datasets, lists, dictionaries and tuples were used. Tuples were used for finalised datasets and lists were used wherever mutability was a requirement.

#### Algorithms
+ The algorithm can run in 3.5 seconds on an Apple M2 with 25GiB memory and 9.49 seconds on a i9-12900H processor with 32GB memory.
+ Several nested FOR loops were required due to the nature of the overall dataset (several TGs and subgroups with categories and subcategories within them). They were also necessary in designing intermediate data structures to parcel and send information across functions. However, they were avoided and reduced everywhere possible.
+ Every function carries out its process in a step-by-step fashion.
+ The algorithm is generalisable for other similar datasets.
+ The overall flowchart is given below.

In [ ]:
# Sourced from Calplus (https://github.com/Calplus)
display(SVG(filename='assets/main.svg'))

### Other Design Decisions

#### Why did we decide to make the code compact?
Considering the length and complexity of our program, we used methods such as 

1) Simple list comprehensions:
<code>result = [x for x in iterable if condition]</code>

2) Lambda functions <code>lambda arguments: expression</code>

3) Additional arguments like key and reverse in <code>sorted()</code> to reverse sorting and to sort by a specific element in within the provided list/tuple, if any.

in order to make the code look simpler and more readable. Although clarity is not sacrificed, it may be difficult for beginners to read the code that combines 2-3 of the above methods.

### Data Unpacking & Preprocessing

We are cleaning up the data given to us: removing all the unnessecary text in the .csv file.
As well as creating a dictionary containing the category name and information.

In [ ]:
# Sourced from Calplus (https://github.com/Calplus)
def datasetGenerator(dataName): 

    """ 
        Generates datasets from .csv files.
        
        Args:
            dataName: File name of dataset.
            
        Returns:
            categories: Cleaned dataset in the form of a dictionary with categories.
    """
    
    # try to open file and parse lines
    try:
        with open(dataName, 'r') as file:
            lines = file.readlines()
        pass
    except FileNotFoundError: 
        print("File not found!")

    # separate body of data from headers
    dataBody = lines[1:]

    # clean up data values (select all values separated by \n and ,)
    cleanData = []
    for i in dataBody:
        cleanData.append(re.findall(r'[^,\n]+', i))

    # clean up headers (select all values separated by \ufeff, \n and ,)
    headers = re.findall(r'[^\ufeff\n,]+', lines[0])

    # instantiate and populate an empty dictionary with the headers as keys and empty arrays for values
    categories = {name: [] for name in headers}

    # clean up headers and flatten all values in the cleanData matrix to make it an array
    values = [value for valueList in cleanData for value in valueList]

    # for each category and its index i (obtained by enumerating the keys in categories), 
    # find every len(categories)-th element from the i-th position in categories.
    # append ALL these values to the arrays within the categories dictionary

    # in the new dictionary, we will add the category from headers and and values from dataBody
    for i, category in enumerate(categories.keys()):
        categories[category].extend(values[i::len(categories)])

    return categories

# call the function with the filename (note that the dataset should be in the same directory as the project)
recordsDataset = datasetGenerator('records.csv')

if verbose:
    print(recordsDataset)

In [ ]:
# Sourced from Calplus (https://github.com/Calplus)
# initialise the mapping for school identifiers (0 - 17)
schoolMapping = {school: itemID for itemID, school in enumerate(set(recordsDataset['School']))}

### Determining TGroup Characteristics

From the data that is reorganised into the dictionary, it can be seen that the unevenness of the data is the biggest issue.  If we had equal amounts of data, we could've split everyone into groups without a hassle! Since there's a discrepancy, though, we have to resort to prioritising certain categories over others. For example, there are different numbers of students in the different schools.

We will calculate the frequency of the data in each category and use it to compute the standard deviation. Later, the standard deviation is used to measure the uneveness in each category for each Tgroup. The higher the standard deviation, the more uneven the data. Hence, we prioritise that category more.

In [ ]:
# Sourced from Calplus (https://github.com/Calplus)
display(SVG(filename='assets/fetch-t-group-info.svg'))

#### Extracting Information from Dataset

We are encoding our data, since numbers are easier to work with than strings.
In this case: 
For gender: Male = '-1' and Female = '1'
For Schools: 0-17 for each School
For cGPA: Above median = '1' and Below median = '-1'

In [ ]:
# Sourced from Calplus (https://github.com/Calplus)
def encodeDataset(limit):
    
    """ 
        Encodes each category in the dataset using numerical values for ease of comparison.
        
        Args:
            limit: Maximum number of people in each tutorial group / limit for initial tutorial group.
            
        Returns:
            genderItems: List containing Gender items sorted into Female or Male using 1 and -1.
            mappedSchoolItems: List containing School items from 0 - 17.
            processedCgpaItems: List containing CGPA items sorted into two bands (low or high scorers) using 1 and -1.
    """
    
    # encode genders as -1 or 1 (each TGroup, 50 students) and record the frequency of each gender
    genderItems = [1 if item == "Female" else -1 for item in recordsDataset['Gender'][(limit-50):limit]]

    # encode schools from 0 to 17 (each TGroup, 50 students) and record the frequency of each school
    schoolItems = recordsDataset['School'][(limit-50):limit]
    mappedSchoolItems = [schoolMapping[item] for item in schoolItems]

    # encode low scorers as -1 and high scorers as 1 (each TGroup, 50 students) and record the frequency of each type of scorer
    cgpaItems = list(map(float, recordsDataset['CGPA'][(limit-50):limit]))
    cgpaMedian = statistics.median(cgpaItems)
    processedCgpaItems = [1 if item > cgpaMedian else -1 for item in cgpaItems]
    
    return genderItems, mappedSchoolItems, processedCgpaItems

In [ ]:
# Sourced from Calplus (https://github.com/Calplus)
def createFullDataset(limit):

    """ 
        Creates a full dataset using encoded datasets.
        
        Args:
            limit: Maximum number of people in each tutorial group / limit for initial tutorial group.
            
        Returns:

            fullyEncodedDataset: Dataset (3D array) containing all category frequencies.
    """

    # obtain encoded datasets as dictionaries
    genderDataset, schoolDataset, cgpaDataset = encodeDataset(limit)
    
    # combine all dictionaries of items into a 3-dimensional array
    fullyEncodedDataset = []
    for i in range(len(genderDataset)):
        fullyEncodedDataset.append([genderDataset[i], schoolDataset[i], cgpaDataset[i]])

    # return frequency counts of all categories as well as the 3-dimensional dataset containing them
    return [fullyEncodedDataset]

In [ ]:
# Sourced from Calplus (https://github.com/Calplus)
def computeFrequenciesOfDictionary(dictionary):
    
    """ 
        Counts the number of times a unique value appears in a category.
        
        Args:
            dictionary: Dictionary containing values to be counted.
            
        Returns:
            frequencies: Dictionary with the same keys as the input but frequencies as values.
    """
    # compute frequency of dictionary by counting the unique values present
    frequencies = {key: dictionary.count(key) for key in set(dictionary)}
    
    return frequencies

In [ ]:
# Sourced from Calplus (https://github.com/Calplus)
def countCategoricalFrequencies(genderItems, schoolItems, cgpaItems):
    
    """ 
        Provides frequency datasets for genders, schools and CGPA.
        
        Args:
            genderItems: Encoded gender dataset.
            schoolItems: Encoded school dataset.
            cgpaItems: Encoded CGPA dataset.
            
        Returns:
            numGenderItems: Dictionary containing frequency of genders.
            numSchoolItems: Dictionary containing frequency of 18 different schools.
            numCgpaItems: Dictionary containing frequency of low scorers and high scorers (relative to median).
    """

    numGenderItems = computeFrequenciesOfDictionary(genderItems)
    numSchoolItems = computeFrequenciesOfDictionary(schoolItems)
    numCgpaItems = computeFrequenciesOfDictionary(cgpaItems)
    
    return numGenderItems, numSchoolItems, numCgpaItems

#### Creating Priority List

Using the frequency calculated for each catergory: numGenderItems, numSchoolItems, numCgpaItems, we will calculate the standard deviation.
Afterwards, the category with the highest standard deviation (most uneven) will be the highest priority as we want to distribute these students evenly across all the groups. The priority list will be sorted from lowest to highest.

In [ ]:
# Sourced from Calplus (https://github.com/Calplus)
checkSchoolStddevs = {}
checkCGPAStddevs = {}
checkGenderStddevs = {}

def prioritiser(limit):
    
    """ 
        Provides an ordered categorical priority list based on standard deviations.
        
        Args:
            limit: Maximum number of people in each tutorial group / limit for initial tutorial group.
            
        Returns:
            sortedStdDevTuples: A sorted array containing tuples with standard deviations and their identifiers.
    """
    
    # obtain encoded datasets and categorical frequencies
    genderDataset, schoolDataset, cgpaDataset = encodeDataset(limit)
    numGenderItems, numSchoolItems, numCgpaItems = countCategoricalFrequencies(genderDataset, schoolDataset, cgpaDataset)
    
    # calculate population standard deviation of frequencies within each category to estimate the spread of data
    # the stddev will be used to figure out which category is the most uneven in terms of frequencies
    if tutGroup not in checkSchoolStddevs:
            checkGenderStddevs[tutGroup] = []
            checkCGPAStddevs[tutGroup] = []
            checkSchoolStddevs[tutGroup] = []
        
    cgpaStdDev = statistics.pstdev(list(numCgpaItems.values()))
    checkCGPAStddevs[tutGroup].append(cgpaStdDev)
    schoolStdDev = statistics.pstdev(list(numSchoolItems.values()))
    checkSchoolStddevs[tutGroup].append(schoolStdDev)
    genderStdDev = statistics.pstdev(list(numGenderItems.values()))
    checkGenderStddevs[tutGroup].append(genderStdDev)

    # store the stddevs as tuples along with their String identifier so they can be accessed easily later on
    stdDevTuples = [[cgpaStdDev, "CGPA"], [schoolStdDev, "School"], [genderStdDev, "Gender"]]
    sortedStdDevTuples = sorted(stdDevTuples, key=lambda x: x[0])
    
    # return "priorities" or standard deviations 
    return sortedStdDevTuples

#### Call Function & Display Results

Now we will print the details for each TGroup:
1) The Tutorial Group number
2) The index assigned to each School
3) The Priority list : from Highest to Lowest Priority

As well as fetching the Tutorial Group information: genderFrequency, schoolFrequency,cgpaFrequency, the full dataset, limit for each TG and priority tuples.

In [ ]:
# Sourced from Calplus (https://github.com/Calplus)
def printTutorialGroup(priorityTuples):
    
    """ 
        Prints details about the given tutorial group
        
        Args:
            priorityTuples: Priority list for the three categories created by reversing the the sorted stdev tuples.

    """
    
    print(f"\n")
    print(f"{BLUE}TUT GROUP: {tutGroup}{RESET}")
    print(f"{BLUE}School Encoding/Mapping:{RESET} {schoolMapping}")

    # data for the priority list and stddev values
    table_data = [
        ["Highest Priority", priorityTuples[-1][1], priorityTuples[-1][0]],
        ["Medium Priority", priorityTuples[-2][1], priorityTuples[-2][0]],
        ["Lowest Priority", priorityTuples[-3][1], priorityTuples[-3][0]],
    ]

    # plot a table
    fig, axis = plt.subplots(figsize=(6, 1))
    axis.axis("off")
    table = axis.table(cellText=table_data, colLabels=["Priority Level", "Description", "Standard Deviation"], loc="center", cellLoc="center", colColours=["#f0f0f0"]*3)
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.2, 1.2)
    plt.show()

In [ ]:
# Sourced from Calplus (https://github.com/Calplus)
def fetchTGroupInfo():
    
    """ 
        Fetches all the information about a tutorial group (frequency datasets, full dataset, limit for each TG, and priority tuples.
        
        Returns:
            genderFrequency: Gender frequency dataset.
            schoolFrequency: School frequency dataset.
            cgpaFrequency: CGPA frequency dataset.
            fullSet: Full dataset.
            index: Index of parent FOR loop.
            priorityTuples: Priority list for the three categories created by reversing the the sorted stdev tuples.

    """
    
    # calculate limit for each tutorial group (eg. 0 to 50, then 50 to 100, etc)
    limit = 50 * (index + 1)

    # obtain encoded datasets and categorical frequenies
    genderSet, schoolSet, cgpaSet = encodeDataset(limit)
    genderFrequency, schoolFrequency, cgpaFrequency = countCategoricalFrequencies(genderSet, schoolSet, cgpaSet)
    
    # obtain priority list
    priorityTuples = prioritiser(limit)

    # create full dataset and print tgroup details
    fullSet = createFullDataset(limit)
    printTutorialGroup(priorityTuples)

    return genderFrequency, schoolFrequency, cgpaFrequency, fullSet, index, priorityTuples

### Searching & Matching Students

In [ ]:
# Sourced from Calplus (https://github.com/Calplus)
display(SVG(filename='assets/search-match.svg'))

#### Calculating Probabilities

We are calculating the probability, so for each item, it is divided by 50. 
The higher the probability, the more common that item is in its category (ie. if there are more Females than Males, then probability for females would be higher). Hence, the maximum probability will be prioritised, they are the ideal person we want MORE OF in a subgroup.

In [ ]:
# Sourced from Calplus (https://github.com/Calplus)
display(SVG(filename='assets/prob-calculator.svg'))

In [ ]:
# Sourced from Calplus (https://github.com/Calplus)
def probabilityCalculator(genderDataset, schoolDataset, cgpaDataset):
    
    """ 
        Calculates probabilities or " priorities".
        
        Args:
            genderDataset:Encoded School dataset.
            schoolDataset: Encoded Gender dataset.
            cgpaDataset: Encoded CGPA dataset.

        Returns:
            GenderProbabilities: Probabilities of unique Gender values based on their frequency in the dataset.
            SchoolProbabilities: Probabilities of unique School values based on their frequency in the dataset.
            CgpaProbabilities: Probabilities of unique CGPA values based on their frequency in the dataset.
            
    """

    # calculate probability scores for each category
    genderProbabilities = {gender: item / 50 for gender, item in genderDataset.items()}
    schoolProbabilities = {school: item / 50 for school, item in schoolDataset.items()}
    cgpaProbabilities = {cgpa: item / 50 for cgpa, item in cgpaDataset.items()}
    
    # display probabilities for reference
    print(f"{BLUE}Gender Probabilities:{RESET} {genderProbabilities}")
    print(f"{BLUE}School Probabilities:{RESET} {schoolProbabilities}")
    print(f"{BLUE}CGPA Probabilities:{RESET} {cgpaProbabilities} \n")
    print(f"\n")
    
    return genderProbabilities, schoolProbabilities, cgpaProbabilities

#### Generate Requirements

In [ ]:
# Sourced from Calplus (https://github.com/Calplus)
display(SVG(filename='assets/gen-requirements.svg'))

First, we need to find the most "suitable" student that fits all 3 categories. For that, within each category, we find the key with the highest value assigned to it.

For example, under the "Gender" category, if Female = 0.60 and Male = 0.40, "Female" have the highest value assigned to it and is therefore the highest priority.

The same steps are repeated for the other 2 categories (School, CGPA). This is considered the highest priority requirement.

In [ ]:
# Sourced from Calplus (https://github.com/Calplus)
def createFirstRequirement(genderProbabilities, schoolProbabilities, cgpaProbabilities):
    
    """ 
        Calculates probabilities or " priorities".
        
        Args:
            genderProbabilities: Probabilities of unique Gender values based on their frequency in the dataset.
            schoolProbabilities: Probabilities of unique School values based on their frequency in the dataset.
            cgpaProbabilities: Probabilities of unique CGPA values based on their frequency in the dataset.

        Returns:
            requirement: The ideal person we want more of in a subgroup.
            
    """

    # generate initial requirement using the highest probabilities
    requirement = [
        max(genderProbabilities, key=genderProbabilities.get),
        max(schoolProbabilities, key=schoolProbabilities.get),
        max(cgpaProbabilities, key=cgpaProbabilities.get)
    ]
    
    return requirement

Once that is found, we need to find the remaining highest priority requirement.

But first, the data needs to be parsed for computation in the next function by assigning each category to an index, and sorting that index from highest to lowest probability.

In [ ]:
# Sourced from Calplus (https://github.com/Calplus)
def setupRequirements(genderScore, schoolScore, cgpaScore, priorityTuples):
    
    """ 
        Sets up necessary data structures for requirement generation and calls said generation function.
        
        Args:
            priorityTuples: Priority list for the three categories created by reversing the the sorted stdev tuples.
            genderScore: Dictionary containing probabilities of unique Gender values based on their frequency in the dataset.
            schoolScore: Dictionary containing probabilities of unique School values based on their frequency in the dataset.
            cgpaScore: Dictionary containing probabilities of unique CGPA values based on their frequency in the dataset.

        Returns:
            requirementResults: Complete and final list of requirements for a tutorial group.
            
    """

    # create initial requirement
    initialRequirement = createFirstRequirement(genderScore, schoolScore, cgpaScore)

    # create lists ordered by probability score
    genderList = [dictKey for dictKey, value in sorted(genderScore.items(), key=lambda x: x[1], reverse=True)]
    schoolsList = [dictKey for dictKey, value in sorted(schoolScore.items(), key=lambda x: x[1], reverse=True)]
    cgpaList = [dictKey for dictKey, value in sorted(cgpaScore.items(), key=lambda x: x[1], reverse=True)]

    # create priority to list and index map so that the requirement generator can follow a given blueprint
    priorityNameToIndexAndListMap = {"Gender": [0, genderList], "School": [1, schoolsList], "CGPA": [2, cgpaList]}

    # generate requirements!
    requirementResults = generateRequirements(priorityTuples, priorityNameToIndexAndListMap, initialRequirement.copy(), level=-1)
    
    return requirementResults

After that, we can start generating the second highest priority requirement.

To do that, we take the category with the lowest priority (standard deviation) and find the second highest value associated with it. For CGPA/Gender, this would be the inverse of that category. i.e: If "Female" is the lowest priority and it was selected earlier, now "Male" would be selected while the other categories remain the same. This counts as the second highest priority.

Once all options in the lowest priority are exhausted (this particularly applies to schools), the second highest priority will be changed. These steps will be repeated until all options are exhausted.

A good way to think about it is a decreasing binary number, starting from [111], then [110], [101], [100] and so on till it reaches [000], except "schools" have 18 values instead of 2.

In [ ]:
# Sourced from Calplus (https://github.com/Calplus)
def generateRequirements(priorityTuples, priorityNameToIndexAndListMap, requirement, level=-1, tutGroupRequirements=None):
    
    """ 
        Recursively generates requirements.
        
        Args:
            priorityTuples: Priority list for the three categories created by reversing the the sorted stdev tuples.
            priorityNameToIndexAndListMap: Maps prioritised category to the list of unique values (sorted based on probability).
            requirement: Current requirement to follow and edit hierarchically.
            level: Priority level to descend from.
            tutGroupRequirements: Final list of ordered requirements
            
        Returns:

            tutGroupRequirements: Ordered list of requirements for the whole tutorial group.
    """

    # instantiate new results list if results list is None
    if tutGroupRequirements is None:
        tutGroupRequirements = []

    # once all the priorities have been handled from the highest priority to the lowest, append the completed requirement
    if level == -4:
        tutGroupRequirements.append(requirement.copy())
        return tutGroupRequirements

    # determine which index and list to use to format the requirement
    indexToUse = priorityNameToIndexAndListMap.get(priorityTuples[level][1])[0]
    listToUse = priorityNameToIndexAndListMap.get(priorityTuples[level][1])[1]

    # assign the item in the list to the designated position in the requirement
    # call the function recursively until all "levels" have been descended
    for itemInList in listToUse:
        requirement[indexToUse] = itemInList
        generateRequirements(priorityTuples, priorityNameToIndexAndListMap, requirement, level - 1, tutGroupRequirements)
        
    return tutGroupRequirements

#### Filter According to Requirements

Once all the categories are identified, the students are then assigned to the categories they fall in. The following chunk of code loops over 50 entries at a time in the bigger dataset, writes details for each tutorial group, and sets off a chain of events by calling ScoreAssigner() (which will go on to start forming groups later on)

In [ ]:
# Sourced from Calplus (https://github.com/Calplus)
display(SVG(filename='assets/find-perf-students.svg'))

In [ ]:
# Sourced from Calplus (https://github.com/Calplus)
display(SVG(filename='assets/return-target-pool.svg'))

In [ ]:
# Sourced from Calplus (https://github.com/Calplus)
def returnTargetPool(items, requirement):
    
    """ 
        Using selected priority, this function identifies members of student pool that possess required characteristics
        
        Args:
            items: Dataset containing pool of students 
            requirement: List of requirements for next member.
            priority: Current category to be prioritised.

        Returns:
            [filtered]: Filtered results nested in an array.
    """
    
    # check if each characteristic for each member matches our requirement and create a list called filtered
    filtered = [filteredItem for filteredItem in items[0] if filteredItem[1:] == requirement]

    # if no matches are found, return same list of items that were provided
    if not filtered:
        return []

    return [filtered]

In [ ]:
# Sourced from Calplus (https://github.com/Calplus)
display(SVG(filename='assets/find-perf-student-func.svg'))

In [ ]:
# Sourced from Calplus (https://github.com/Calplus)
def findPerfectStudents(targetPool, requirement, requirementsList, idxForLimit):
    
    """ 
        Identifies best candidate for a group using the available pool of students and requirements.
        
        Args:
            targetPool: Full dataset to enumerate, search and shorten as much as possible
            requirement: Current requirement.
            requirementsList: List of requirements for a tutorial group.
            idxForLimit: Index of tutorial group starting from 0 to calculate limit index.
    """
    
    
    # filter for the perfect candidates
    targetPool = returnTargetPool(targetPool, requirement)
             
    if targetPool:
        if tutGroup not in matchedStudents:
            matchedStudents[tutGroup] = []
        if verbose:
            # print matches
            print(f"Using the above requirement, the following students were found: {GREEN}{targetPool}{RESET}\n")
        
        #find exact student id
        for student in targetPool[0]:
            student_id = student[0]
            studentName = recordsDataset['Name'][student[0]]
            matchedStudents[tutGroup].append((student_id, studentName, requirementsList.index(requirement)))
    else:
        if verbose:
            print(f"{RED}No students meeting the above requirement were found!{RESET}\n")


This function assigns each student in the data list their own index, with the first TG being students 1-50, second being 51-100, etc. for easier matching later on.

In [ ]:
# Sourced from Calplus (https://github.com/Calplus)
def indexStudentPool(givenPool, idxForLimit):
    
    """ 
        Indexes students from 1-50, 51-100, and so on...
        
        Args:
            givenPool: Given dataset.
            idxForLimit: Index of tutorial group starting from 0 to calculate limit index.
            
        Return:
            indexedStudentPool: Indexed list of students.
    """

    # index students from 1 to 6000 by adding a new element at the 0th position
    indexedStudentPool = [[[index + idxForLimit * 50] + student for index, student in enumerate(students)] for students in givenPool]
    return indexedStudentPool

Once all the students within the TGroup are sorted, they are merged into 1 list, taking students from the highest priority requirement and descending to the lowest priority requirement.

In [ ]:
# Sourced from Calplus (https://github.com/Calplus)
def searchMatchStudents(genderDataset, schoolDataset, cgpaDataset, fullDataset, idxForLimit, priorityTuples):

    """ 
        Sets " scores" using probabilities for each category and creates requirements for one group. Then it begins the search.
        
        Args:
            genderItems: Gender dataset,
            schoolItems: School dataset.
            cgpaItems: CGPA dataset.
            fullDataset: Full dataset.
            idxForLimit: Index of tutorial group starting from 0 to calculate limit index.
            priorityTuples: Priority list for the three categories created by reversing the the sorted stdev tuples.
    """

    # create dictionaries containing probabilities for each categorical frequency
    GenderScore, SchoolScore, CgpaScore = probabilityCalculator(genderDataset, schoolDataset, cgpaDataset)

    # construct a requirement list based on the maximum probability in each categorical dictionary
    requirementsList = setupRequirements(GenderScore, SchoolScore, CgpaScore, priorityTuples)
    
    # indexes students to facilitate retrieval
    indexedStudentPool = indexStudentPool(fullDataset, idxForLimit)

    # adds all the data to a new dataset for graphing purposes
    addToGraphingDataset(indexedStudentPool, tutGroup)
    
    # time to look for a student using the requirements!
    for currentRequirement in requirementsList:
        if verbose:
            print("Looking for:", currentRequirement)
        findPerfectStudents(indexedStudentPool, currentRequirement, requirementsList, idxForLimit) # initiates search for students

The next line of code is for graphing the data obtained.

In [ ]:
# Sourced from Calplus (https://github.com/Calplus)
graphingDataset = {}

def addToGraphingDataset(graphData, tutGroup):
        
    """ 
        Adds data to a dictionary called graphingDataset for graphing purposes.
        
        Args:
            graphData: Encoded and indexed student dataset.
            tutGroup: Current tutorial group name.
            
    """
    # create new dataset for graphing purposes
    graphData = [item for sublist in graphData for item in sublist]
    graphingDataset[tutGroup] = graphData


In [ ]:
# Sourced from Calplus (https://github.com/Calplus)
matchedStudents = {}

# iterate through tutorial groups and match students using requirements
for index, tutGroup in enumerate(list(dict.fromkeys(recordsDataset["Tutorial Group"]))):
    genderFrequency, schoolFrequency, cgpaFrequency, fullSet, index, priorityTuples = fetchTGroupInfo()
    # print ("gender frequency:", genderFrequency)
    # print ("school frequency:", schoolFrequency)
    # print ("cgpa frequency:", cgpaFrequency)
    searchMatchStudents(genderFrequency, schoolFrequency, cgpaFrequency, fullSet, index, priorityTuples)

### Assignment Using Calvin Grid

And now to tie everything together into the focus of this project: The assignment algorithm.

When people think about their assignment algorithm, their first thought is to find the first perfect group of 5 to form the most diverse group, before repeating the process for all the remaining groups. The problem with this approach, however, is that while the first group will be the most diverse, any subsequent groups will become less diverse as the most unique people get picked out of the pool of remaining students, leaving the more generic students behind.

For example, if you have 40 females and 10 males, assuming a "perfect group" consists of 3 females and 2 males, only 5 groups will meet that criteria. However, you are then left with 25 females for 5 groups, meaning the remaining groups will only have 5 females. Obviously this algorithm would not work, and you can tell from first glance that an evenly divided group should have 4 females and 1 male.

This is where we have to change our thought process. We have to accept that we will never be able to make every group perfectly diverse. We will either have a few perfectly diverse groups and a few generic groups (which was the aforementioned algorithm), or we can try to balance the diversity across all groups, regardless of how many "perfectly diverse" groups there will be. So we chose the latter option.

To do that, rather than finding the most unique students, we want to find the most "generic" group of students and divide them evenly across all the groups (1 student/group). Then we move on to the 2nd most generic group and continue dividing them evenly. If there are any leftovers, the next group will continue where the previous group left off, and once each group has 1 student, we will loop back to the 1st group and continue sorting students. This way, we reduce the probability someone of the same category will be in the same group, therefore making each group as diverse as possible, given the students.

In [ ]:
# Sourced from Calplus (https://github.com/Calplus)
display(SVG(filename='assets/calvin-grid.svg'))

In [ ]:
# Sourced from Calplus (https://github.com/Calplus)
def calvinGrid(orderedStudentList, tGroup):
        
    """ 
        Arranges students into a 5x10 grid for each tutorial group.
        
        Args:
            orderedStudentList: Ordered list of students based on requirements (perfect matches in the start).
            key: Tutorial group name.
    """

    # arrange students horizontally across Calvin grid
    for i in range(5): # 5 rows
        for j in range(0, 10): # 10 columns
            groupedStudents[tGroup][j].append(orderedStudentList[10*i + j])

To demonstrate how our algorithm works, using the previous example (40 females, 10 males), we will assume that "gender" is the category with the highest standard deviation. So the algorithm will prioritise sorting gender first. It will then take the gender with the highest count, in this case, females, and sort them first. That way, each group will have 4 females. We still have 2 other categories: CGPA and Schools. So we then take the category with the 2nd highest standard deviation and sort the students accordingly, and then repeating the process for the last category. With this information, we can create an ordered list where the most "generic" student will be first in the list while the least "generic" student is last. That is what all the code above this does.

With this ordered list, calvingrid() will assign each student a number from 0-9, in ascending order. Once the 10th student in the list is assigned a number (9), it will repeat the process 4 more times (so the next student is assigned the number "0").

groupstudents() then takes all the people assigned to the same number, and make them form a group.

In [ ]:
# Sourced from Calplus (https://github.com/Calplus)
def groupStudents():
        
    """ 
        Groups students using the Calvin grid by passing in the full dataset and stores the results in groupedStudents.
        
        Returns:
            orderedStudentList: Ordered list of students based on requirements (perfect matches in the start).
    """
    
    # initialise global groupedStudents dictionary
    global groupedStudents
    groupedStudents = {tGroup: [[] for _ in range(10)] for tGroup in matchedStudents}
    
    for tGroup in matchedStudents:
        
        # get list of students for tGroup
        orderedList = matchedStudents.get(tGroup)
        if verbose:
            print(f"{BOLD}Processing group {tGroup} with ordered list:{RESET}\n")
            print(f"{orderedList}\n")
            print(f"\n")

        # run the calvin grid
        calvinGrid(orderedList, tGroup)

groupStudents()

### Results Analysis

Here, we set up all the functions required to start graphing the results.

In [ ]:
# Sourced from Calplus (https://github.com/Calplus)
# cute pastel colour scheme :)
colors = ['#FBB4B9', '#FDD0A2', '#FBE7A1', '#B3E2CD', '#CCEBC5', '#D9E9D5', '#F7C1BE', '#F3B9D3', '#FFCC99', '#D4F1C2', '#E9D0D9', '#B8D4E1', '#E1F7D4', '#C7B7F6', '#D6F1FF', '#C6D7F7']

In [ ]:
# Sourced from Calplus (https://github.com/Calplus)
def matchMemberToRecords(studentIds, fullDataset):
            
    """ 
        Find a group of students' characteristics by matching them using their ID.
        
        Returns:
            matchedGenderItems: Encoded gender data for desired students.
            matchedSchoolItems: Encoded school data for desired students.
            matchedCgpaItems: Encoded CGPA data for desired students.
    """
    
    matchedGenderItems = [record[1] for record in fullDataset if record[0] in studentIds]
    matchedSchoolItems = [record[2] for record in fullDataset if record[0] in studentIds]
    matchedCgpaItems = [record[3] for record in fullDataset if record[0] in studentIds]
    
    return matchedGenderItems, matchedSchoolItems, matchedCgpaItems

In [ ]:
# Sourced from Calplus (https://github.com/Calplus)
def fetchStudentIDsInGroup(groupData):
                
    """ 
        Get student IDs for a given group of students.

        Args:
            groupData: Dataset containing students.
        
        Returns:
            studentIds: List of student IDs for a certain group.
    """

    # use list comprehension
    studentIds = [student[0] for student in groupData]
    return studentIds

In [ ]:
# Sourced from Calplus (https://github.com/Calplus)
def extractDistributions(groupData, fullDataset):
                    
    """ 
        Extract distributions of unique values from given data by first finding the correct students with their IDs and 
        then computing the frequencies.

        Args:
            groupData: Desired dataset containing students.
            fullDataset: Entire student dataset.
        
        Returns:
            genderCount: Dictionary containing frequencies of unique Gender values.
            schoolCount: Dictionary containing frequencies of unique School values.
            cgpaCount: Dictionary containing frequencies of unique CGPA values.
    """

    studentIds = fetchStudentIDsInGroup(groupData)
    matchedGenderItems, matchedSchoolItems, matchedCgpaItems = matchMemberToRecords(studentIds, fullDataset)

    genderCount = computeFrequenciesOfDictionary(matchedGenderItems)
    schoolCount = computeFrequenciesOfDictionary(matchedSchoolItems)
    cgpaCount = computeFrequenciesOfDictionary(matchedCgpaItems)
    
    return genderCount, schoolCount, cgpaCount

In [ ]:
# Sourced from Calplus (https://github.com/Calplus)
def findRatiosForGroup(genderFreq, cgpaFreq, schoolFreq):
    
    # express ratios in string format
    genderRatio = f"{max(genderFreq.values())}-{min(genderFreq.values())}"
    cgpaRatio = f"{max(cgpaFreq.values())}-{min(cgpaFreq.values())}"
    schoolRatio = "-".join(map(str, sorted(schoolFreq.values(), reverse=True)))
    
    return genderRatio, cgpaRatio, schoolRatio

In [ ]:
# Sourced from Calplus (https://github.com/Calplus)
def matchesCriteria(genderDist, schoolDist, cgpaDist):
                        
    """ 
        Retrieve ratios of distributions for each category (60-40s and 20-20-20-20-20s).

        Args:
            genderDist: Dictionary containing frequencies of unique School values.nary containing frequencies of unique Gender values.
            schoolDist: Dictionary containing frequencies of unique School values.
            cgpaDist: Dictionary containing frequencies of unique CGPA values.
        
        Returns:
            genderCriteria: Boolean results for whether Gender entries meet the 60-40 criterion.
            schoolCriteria: Boolean results for whether School entries meet the 60-40 criterion.
            cgpaCriteria: Boolean results for whether CGPA entries meet the 20-20-20-20-20 criterion. 
    """

    # boolean expressions for 3-2 and 2-3 ratios for gender
    genderCriteria = (
        (genderDist.get(1, 0) == 3 and genderDist.get(-1, 0) == 2) or
        (genderDist.get(1, 0) == 2 and genderDist.get(-1, 0) == 3)
    )

    # boolean expressions for 3-2 and 2-3 ratios for cgpa
    cgpaCriteria = (
        (cgpaDist.get(1, 0) == 3 and cgpaDist.get(-1, 0) == 2) or
        (cgpaDist.get(1, 0) == 2 and cgpaDist.get(-1, 0) == 3)
    )

    # boolean expressions for 1-1-1-1-1 ratios for school
    schoolCriteria = all(count == 1 for count in schoolDist.values())
    
    return genderCriteria, cgpaCriteria, schoolCriteria

In [ ]:
# Sourced from Calplus (https://github.com/Calplus)
allThreeCriteriaCount = 0
onlyTwoCriteriaCount = 0
exactlyOneCriterionCount = 0

In [ ]:
# Sourced from Calplus (https://github.com/Calplus)
def printSubroupDetails():
                            
    """ 
        Print raw data about met criteria for each tutorial group.

    """
    
    print(f"Tutorial Group {group}:")
    print(f"  60-40 Gender Distribution Subgroups: {gender60_40Count} out of 10")
    print(f"  60-40 CGPA Distribution Subgroups: {cgpa60_40Count} out of 10")
    print(f"  20-20-20-20-20 School Distribution Subgroups: {school20_20_20_20_20Count} out of 10\n")

In [ ]:
# Sourced from Calplus (https://github.com/Calplus)
def printOverallResults():
                                
    """ 
        Print overall results about met criteria for each tutorial group.

    """
    print(f"{BOLD}Number of subgroups meeting all three criteria:{RESET} {allThreeCriteriaCount} out of 1200")
    print(f"{BOLD}Number of subgroups meeting exactly two criteria:{RESET} {onlyTwoCriteriaCount} out of 1200")
    print(f"{BOLD}Number of subgroups meeting exactly one criterion:{RESET} {exactlyOneCriterionCount} out of 1200")
    print(f"{BOLD}Number of subgroups that meet no criteria:{RESET} {1200 - exactlyOneCriterionCount - onlyTwoCriteriaCount - allThreeCriteriaCount} out of 1200\n")

Count all the subgroups that meet the following 60-40 or 20-20-20-20-20 criteria.

In [ ]:
# Sourced from Calplus (https://github.com/Calplus)
for group, studentsInCurrentGroup in groupedStudents.items():
    gender60_40Count, cgpa60_40Count, school20_20_20_20_20Count = 0, 0, 0
    for studentsSubgroup in studentsInCurrentGroup:

        # obtain category distributions in each subgroup and booleans for criteria
        genderDist, schoolDist, cgpaDist = extractDistributions(studentsSubgroup, graphingDataset[group])
        genderMatch, cgpaMatch, schoolMatch = matchesCriteria(genderDist, schoolDist, cgpaDist)

        # count the number of criteria met for each subgroup by adding ones or zeroes
        gender60_40Count += genderMatch
        cgpa60_40Count += cgpaMatch
        school20_20_20_20_20Count += schoolMatch

        matchCount = genderMatch + cgpaMatch + schoolMatch
        if matchCount == 3:
            allThreeCriteriaCount += 1
        elif matchCount == 2:
            onlyTwoCriteriaCount += 1
        elif matchCount == 1:
            exactlyOneCriterionCount += 1
        if verbose:
            # to print out subgroup details
            printSubroupDetails()

In [ ]:
# Sourced from Calplus (https://github.com/Calplus)
def plotHistogram(counts, binEdges):
                            
    """ 
        Plot a histogram using frequencies via binning

        Args:
            counts:
            countsScaled:
            binEdges:
        
        Returns:
            genderCriteria: Counts of Gender entries that meet the 60-40 criterion.
            schoolCriteria: Counts of School entries that meet the 60-40 criterion.
            cgpaCriteria: Counts of CGPA entries that meet the 20-20-20-20-20 criterion. 
    """
    
    color_step = 1 / len(counts) # color map stepping
    colors = [plt.cm.plasma(i * color_step) for i in range(len(counts))] # gradual shift in color map
    plt.bar(
        binEdges[:-1], # edges of bars (that's why we ignore the last one)
        counts, # height of bar
        width=0.5, 
        align='center', 
        alpha=1, 
        color=colors, 
        edgecolor="black"
    )

    
    for i, count in enumerate(counts):
        if count > 0:
            plt.text(binEdges[i], count, str(int(count)), 
                     ha='center', va='bottom', fontsize=8) # plot each bar
            
    plt.xticks(range(0, 10, 1))
    plt.title("Diversity Bell Curve for Subgroups", fontsize=10)
    plt.xlabel("Diversity Score", fontsize=10)
    plt.ylabel("Frequency", fontsize=10)
    plt.grid(axis="y", alpha=0.5)
    plt.show()


In [ ]:
# Sourced from Calplus (https://github.com/Calplus)
def lowestOrHighestGroupDetails(subgroup, group, score, textInput):   
                                
    """ 
        Print details about lowest subgroup.
    """
    
    print(f"\n{BOLD}Subgroup with the {textInput} score:{RESET}")
    print("\n".join(map(str, subgroup)))
    print(f"\n{BOLD}{textInput} scoring tutorial group:{RESET} {group}")
    print(f"{BOLD}{textInput} diversity score:{RESET} {score}\n")

In [ ]:
# Sourced from Calplus (https://github.com/Calplus)
# score mapping based on ratios
genderScores = {"3-2": 3, "4-1": 2, "5": 1}
cgpaScores = {"3-2": 3, "4-1": 2, "5": 1}
schoolScores = {"1-1-1-1-1": 3, "2-1-1-1": 2.5, "2-2-1": 2, "3-1-1": 1.5, "3-2": 1, "4-1": 0.5, "5": 0}

In [ ]:
# Sourced from Calplus (https://github.com/Calplus)
def mapToScore(genderFreq, cgpaFreq, schoolFreq):
                                
    """ 
        Plot a histogram using frequencies via binning

        Args:
            genderFreq: Distribution of Genders in subgroup.
            cgpaFreq: Distribution of CGPA in subgroup.
            schoolFreq: Distribution of Schools in subgroup.
        
        Returns:
            totalGroupScore: A diversity score (9 being the highest possible value)
    """
    
    # obtain ratios using frequencies
    groupGenderRatio, groupCgpaRatio, groupSchoolRatio = findRatiosForGroup(genderFreq, cgpaFreq, schoolFreq)
    
    # add up each category scores using score mapping
    totalGroupScore = genderScores.get(groupGenderRatio, 0) + cgpaScores.get(groupCgpaRatio, 0) + schoolScores.get(groupSchoolRatio, 0)
    
    return totalGroupScore

#### Plot Histogram

In [ ]:
# Sourced from Calplus (https://github.com/Calplus)
# initialise list for storing individual subgroup scores
subgroupScores = []

# initialise variables for lowest subgroup and group
lowestScore, lowestSubgroup, lowestGroup = float('inf'), None, None

# initialise variables for highest subgroup and group
highestScore, highestSubgroup, highestGroup = float('-inf'), None, None

for group, studentsList in groupedStudents.items():
    for studentsGroup in studentsList:
        # for each subgroup in each TGroup, extract distributions and obtain scores
        genderDist, schoolDist, cgpaDist = extractDistributions(studentsGroup, graphingDataset[group])
        currentGroupScore = mapToScore(genderDist, cgpaDist, schoolDist)
        subgroupScores.append(currentGroupScore)

        # record lowest possible score
        if currentGroupScore < lowestScore:
            lowestScore, lowestSubgroup, lowestGroup = currentGroupScore, studentsGroup, group
            
        # record highest possible score
        if currentGroupScore > highestScore:
            highestScore, highestSubgroup, highestGroup = currentGroupScore, studentsGroup, group

binWidth = 0.5                                                    # bar bin for counting occurences of a certain score
numBins = int((highestScore - lowestScore) / binWidth) + 1               # number of bins
binEdges = [lowestScore + binWidth * i for i in range(numBins + 1)]  # number of bins

counts = [0] * numBins
for score in subgroupScores:
    binIndex = int((score - lowestScore) / binWidth)
    counts[binIndex] += 1

plotHistogram(counts, binEdges)
lowestOrHighestGroupDetails(lowestSubgroup, lowestGroup, lowestScore, "lowest")
lowestOrHighestGroupDetails(highestSubgroup, highestGroup, highestScore, "highest")
printOverallResults()

#### Plot Pie Chart for Lowest Subgroup

In [ ]:
# Sourced from Calplus (https://github.com/Calplus)
def plotPie(axis, data, labels, title, colors=None, textprops=None):
    axis.pie(data, labels=labels, autopct='%1.1f%%', startangle=90, colors=colors, textprops=textprops)
    axis.set_title(title)

lowGenderDist, lowSchoolDist, lowCgpaDist = extractDistributions(lowestSubgroup, graphingDataset[lowestGroup])

fig, axis = plt.subplots(1, 3, figsize=(15, 5))
lowSchoolLabels = list(lowSchoolDist.keys())
lowSchoolSizes = [lowSchoolDist.get(label, 0) for label in lowSchoolLabels]

plotPie(axis[0], [lowGenderDist.get(1, 0), lowGenderDist.get(-1, 0)], ['1', '-1'], "Gender: 1 or -1", ['#FBE7A1', '#B3E2CD'])
plotPie(axis[1], [lowCgpaDist.get(1, 0), lowCgpaDist.get(-1, 0)], ['1', '-1'], "GPA: 1 or -1", ['#FBE7A1', '#B3E2CD'])
plotPie(axis[2], lowSchoolSizes, lowSchoolLabels, "School: 0 - 17", colors)

plt.tight_layout()
plt.show()

#### Plot Pie Chart for Highest Subgroup

In [ ]:
# Sourced from Calplus (https://github.com/Calplus)
fig, axis = plt.subplots(1, 3, figsize=(15, 5))

highGenderDist, highSchoolDist, highCgpaDist = extractDistributions(highestSubgroup, graphingDataset[highestGroup])

highSchoolLabels = list(highSchoolDist.keys())
highSchoolSizes = [highSchoolDist.get(label, 0) for label in highSchoolLabels]

plotPie(axis[0], [highGenderDist.get(1, 0), highGenderDist.get(-1, 0)], ['1', '-1'], "Gender: 1 or -1", ['#FBE7A1', '#B3E2CD'])
plotPie(axis[1], [highCgpaDist.get(1, 0), highCgpaDist.get(-1, 0)], ['1', '-1'], "GPA: 1 or -1", ['#FBE7A1', '#B3E2CD'])
plotPie(axis[2], highSchoolSizes, highSchoolLabels, "School: 0 - 17", colors)

plt.tight_layout()
plt.show()

#### Plot Pie Charts for TGroup Containing Lowest Subgroup

In [ ]:
# Sourced from Calplus (https://github.com/Calplus)
def printAllPieCharts(subgroups, titleText, colors, subgroupIdentifier):
    fig, axes = plt.subplots(10, 3, figsize=(12, 20))
    for i, subgroup in enumerate(subgroups):
        genderDist, schoolDist, cgpaDist = extractDistributions(subgroup, graphingDataset[subgroupIdentifier])
        score = mapToScore(genderDist, cgpaDist, schoolDist)
    
        data = [
            ([genderDist.get(1, 0), genderDist.get(-1, 0)], ['1', '-1'], ['#FBE7A1', '#B3E2CD'], f"Gender (Score: {score})"),
            ([cgpaDist.get(1, 0), cgpaDist.get(-1, 0)], ['1', '-1'], ['#FBE7A1', '#B3E2CD'], f"CGPA (Score: {score})"),
            ([schoolDist.get(label, 0) for label in schoolDist.keys()], list(schoolDist.keys()), colors[:len(schoolDist)], f"School (Score: {score})")
        ]
    
        for j, (sizes, labels, colors, title) in enumerate(data):
            plotPie(axes[i][j], sizes, labels, None, colors, textprops={'fontsize': 8})
            axes[i][j].set_title(f"{title}: Subgroup {i+1}", fontsize=8)
            
    fig.suptitle(f"{titleText}: {lowestGroup}", fontsize=14)
    plt.tight_layout(rect=[0, 0, 1, 0.98])
    plt.show()

In [ ]:
# Sourced from Calplus (https://github.com/Calplus)
printAllPieCharts(groupedStudents[lowestGroup], "Tutorial Group With Lowest Scoring Subgroup", colors, lowestGroup)

In [ ]:
# Sourced from Calplus (https://github.com/Calplus)
printAllPieCharts(groupedStudents[highestGroup], "Tutorial Group With Highest Scoring Subgroup", colors, highestGroup)

#### Export Completed CSV File

In [ ]:
# Sourced from Calplus (https://github.com/Calplus)
if os.path.exists('FCS8-Team1-Group1.csv'):
    os.remove('FCS8-Team1-Group1.csv')
    print(f"File has been deleted to create a new one.")
    
with open('FCS8-Team1-Group1.csv', mode='w') as file:
    file.write('Group,Subgroup,Student ID,Name\n')

    for group, subgroups in groupedStudents.items():
        for i, subgroup in enumerate(subgroups):
            for student in subgroup:
                student_id, name, _ = student
                line = f"{group},Subgroup {i+1},{student_id},{name}\n"
                file.write(line)

print("CSV file has been created!")
completeRuntime = time.time() # stop recording runtime
print("Runtime:", round(completeRuntime - recordRuntime, 2), " second(s)")

In [ ]:
# Sourced from Calplus (https://github.com/Calplus)
print(f"{BOLD}APPENDIX B: USE OF AI TOOL(S) IN PROJECT WORK{RESET}")
todaysDate = time.strftime("%d/%m/%Y", time.localtime())
print("A. I affirm that my contribution(s) to the lab work is my own, produced without help from any AI tool(s)")
print("B. I affirm that my contribution(s) to the lab work has been produced with the help from AI tool(s)")
appendixData = [
    ["[REDACTED MEMBER 1]", todaysDate, "A"],
    ["[REDACTED USER (CALPLUS)]", todaysDate, "A"],
    ["[REDACTED MEMBER 2]", todaysDate, "A"],
    ["[REDACTED MEMBER 3]", todaysDate, "A"],
    ["[REDACTED MEMBER 4]", todaysDate, "A"]
]


fig, axis = plt.subplots(figsize=(6, 1))
axis.axis("off")
table = axis.table(cellText=appendixData, colLabels=["Name", "Date", "A or B"], loc="center", cellLoc="center", colColours=["#f0f0f0"]*3)
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 1.2)
plt.show()
print("Appendix B form is also present in project directory.")